In [1]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch, Rectangle
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from scipy.stats import gaussian_kde
from scipy.stats import chisquare, kruskal, mannwhitneyu, spearmanr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from libpysal.weights import lat2W
from libpysal.weights import Queen
from libpysal.weights import DistanceBand
from esda.moran import Moran_Local, Moran
from shapely.geometry import box

First look at the data

In [2]:
df = pd.read_csv("../CWData_clean7.csv")
pd.set_option("display.max_columns", None)
df.head()

C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\3727079677.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

,id,root_id,category,description,latitude,longitude,spotted_at,spotted_by,spotted_by_type,spotted_by_name,spotted_by_topic_role,created_at,created_at_local,created_by,modified_at,modified_by,date,year_month,Country,Region,City,top_country,top_country_count,total_obs_user,first_country,percent_in_country,Category,Category_Nr,Waterlevel_Virtual,SoilMoisture,SoilMoisture_Nr,TempStream,TempStream_Nr,Plastic_Amount,Plastic_Amount_Nr,Stream_Width,Stream_Depth,Streambed_Material,Streambed_Material_Nr,FlowVelocity_Method,FlowVelocity_Method_Nr,FlowVelocity_direct_vel,FlowVelocity_PS_dist,FlowVelocity_PS_time1,FlowVelocity_PS_time2,FlowVelocity_PS_time3,Plastic_ObservationTime,Plastic_Location,Plastic_Location_Nr,Plastic_RiverWidth,Plastic_RiverWidth_Nr,Plastic_PET,Plastic_PET_Nr,Plastic_POSoft,Plastic_POSoft_Nr,Plastic_POHard,Plastic_POHard_Nr,Plastic_PS,Plastic_PS_Nr,Plastic_PSE,Plastic_PSE_Nr,Plastic_PMultilayer,Plastic_PMultilayer_Nr,Plastic_POther,Plastic_POther_Nr,Plastic_Shore_Plotsize,Plastic_Shore_Plotsize_Nr,Plastic_Removed,Plastic_River_Stagnant,Waterlevel_Physical_Unit,Waterlevel_Physical_Unit_Nr,Waterlevel_Physical,Streamtype,Streamtype_Nr,Swimming_Quality,Drinking_Quality,Naturality,Naturality_Nr,StreamColor,StreamColor_Nr,Stream_Ground_Visibility,Stream_Ground_Visibility_Nr,Stream_Animals,Stream_Pollution_Reason,Stream_sometimes_dry,Stream_Name,TempStream_snow_ice,Stream_Waterquality,Stream_Waterquality_Nr,Stream_Waterclarity,Stream_Waterclarity_Nr,StreamColor_other,Stream_Vegetation,Stream_Foam,Stream_Foam_Nr,Stream_Algae,Stream_Algae_Nr,Stream_Odor,Stream_Odor_Type,Stream_Odor_Type_Nr,Stream_Odor_Type_other,Stream_Litter,Stream_Litter_Nr,Stream_Flow_Alteration,Stream_Flow_Alteration_Nr,Stream_typical_Color,Stream_typical_Color_Nr,Stream_Drainage_Basin,Stream_Drainage_Basin_Nr,Watertype,Watertype_Nr,Lake_Usage,Lake_Usage_Nr,Lake_Access,Lake_Shore_State,Lake_Shore_State_Nr,Lake_Swimming,Lake_Transparency,Lake_Transparency_Nr,Lake_Color,Lake_Color_Nr,Lake_Odor,Lake_Odor_Nr,Lake_Shore_Vegetation,Lake_Shore_Vegetation_Nr,Lake_Underwater_Vegetation,Lake_Floating_Leaveplants,Lake_Duckweed,Lake_Duckweed_Nr,Lake_Mussels,Lake_Mussels_Nr,Lake_Deadwood,Lake_Deadwood_Nr,Lake_Animals,Lake_Animals_Nr,Lake_Waterlevel_Changes,Lake_Waterlevel_Changes_Nr,Lake_Dry,Lake_Dry_Nr,Image_Gallery,image,geo_hash
0,16726,16726,470,Bei der Brücke am Parkplatz der Pizolbahn,47.029225,9.433322,2017-02-05 14:49:14,1996,2,Simon Meili,editor,2017-02-05 13:49:16,2017-02-05 14:49:16,1996,2019-12-29 12:55:32,11596.0,2017-02-05,2017-02,Switzerland,Sankt Gallen,Vaduz,Switzerland,155,246,Switzerland,63.00813,virtual scale,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,River and Land,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17726.0,000007/2017/02/11ba68865090752368e3a24f55058a24,u0qew9mjh6fw
1,16727,16726,470,NaN,47.029225,9.433322,2017-02-05 14:50:00,1996,2,Simon Meili,editor,2017-02-05 13:50:58,2017-02-05 14:50:58,1996,2026-03-19 15:21:53,118221.0,2017-02-05,2017-02,Switzerland,Sankt Gallen,Vaduz,Switzerland,155,246,Switzerland,63.00813,virtual scale,3,0.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,0.05,gravel,2.0,poo-stick,2.0,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,River and Land,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17727.0,000007/2017/02/2f43e32aa5f88ed53822f9a75f9155ea,u0qew9mjh6fw
2,16736,16736,469,NaN,47.398311,8.5

Here, I create an overview map with all the observation points. Locations with more observations get bigger points. 

In [3]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

counts = df.groupby(["latitude", "longitude"]).size().reset_index(name="count")

map_center = [df["latitude"].mean(),df["longitude"].mean()]
m = folium.Map(location=map_center, zoom_start=7)

for _, row in counts.iterrows():
    folium.Circle(
        location = [row["latitude"], row["longitude"]],
        radius = 5 + row["count"] * 2,
        color=  "blue",
        fill = True,
        fill_color = "blue",
        fill_opacity = 0.5
    ).add_to(m)

m

m.save("../Products/Overview.html")

C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\2451309005.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

The same map is produced as a static map

In [4]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])
world = gpd.read_file(f"../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")
world = world.rename(columns={"NAME": "Country"})

counts = df.groupby(["latitude", "longitude"]).size().reset_index(name="count")

gdf_points = gpd.GeoDataFrame(
    counts,
    geometry=gpd.points_from_xy(counts["longitude"], counts["latitude"]),
    crs="EPSG:4326"
).to_crs("+proj=robin")

map_df_all = world.to_crs("+proj=robin")

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

# plot world map
map_df_all.plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

# plot points
gdf_points.plot(
    ax=ax,
    color="teal",
    edgecolor="black",
    linewidth=0.3,
    alpha=0.7,
    markersize=np.log1p(gdf_points["count"]) * 40
)

legend_counts = [1, 10, 50, 100]
legend_handles = [
    plt.scatter([], [], s=np.log1p(c) * 40, color="teal", edgecolor="black", linewidth=0.3, alpha=0.7, label=str(c))
    for c in legend_counts
]
legend = ax.legend(
    handles=legend_handles,
    title="Observations",
    loc="lower left",
    frameon=True,
    labelspacing=1.5,
)
legend.get_title().set_fontsize(15)
for text in legend.get_texts():
    text.set_fontsize(12)

ax.set_title("All Observations", fontsize=18)
ax.set_axis_off()

plt.savefig("../Products/Overview.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\1735340019.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

Then, a bar chart with 3 subplots of number of observations per country, number of users per country and average number of observations per user per country

In [5]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

# data
country_freq = df["Country"].value_counts()
users_per_country = df.groupby("Country")["created_by"].nunique().sort_values(ascending=False)
obs_per_user_country = (country_freq / users_per_country).dropna()

top50_countries = country_freq.head(50).index.tolist()

# obs per country
top50_obs = country_freq[country_freq.index.isin(top50_countries)].reindex(top50_countries)
other_obs = pd.Series({"Other (Average)": country_freq[~country_freq.index.isin(top50_countries)].mean()})
obs_plot = pd.concat([top50_obs, other_obs])

# users per country
top50_users = users_per_country[users_per_country.index.isin(top50_countries)].reindex(top50_countries)
other_users = pd.Series({"Other (Average)": users_per_country[~users_per_country.index.isin(top50_countries)].mean()})
users_plot = pd.concat([top50_users, other_users])

# avg obs per user per country
top50_opu = obs_per_user_country[obs_per_user_country.index.isin(top50_countries)].reindex(top50_countries)
other_opu = pd.Series({"Other (Average)": obs_per_user_country[~obs_per_user_country.index.isin(top50_countries)].mean()})
opu_plot = pd.concat([top50_opu, other_opu])

# plot
fig, axes = plt.subplots(3, 1, figsize=(16, 23), sharex=True)

obs_plot.plot(kind="bar", color="teal", ax=axes[0], logy=True, fontsize=13)
axes[0].set_title("a) Number of Observations per Country", fontsize=21, fontweight="bold", loc="left")
axes[0].set_ylabel("Number of Observations", fontsize=18)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
axes[0].set_axisbelow(True)

users_plot.plot(kind="bar", color="teal", ax=axes[1], logy=True, fontsize=13)
axes[1].set_title("b) Number of Users per Country", fontsize=21, fontweight="bold", loc="left")
axes[1].set_ylabel("Number of Users", fontsize=18)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)
axes[1].set_axisbelow(True)

opu_plot.plot(kind="bar", color="teal", ax=axes[2], fontsize=13)
axes[2].set_title("c) Average Number of Observations per User per Country", fontsize=21, fontweight="bold", loc="left")
axes[2].set_ylabel("Average Number of Observations per User", fontsize=18)
axes[2].grid(axis="y", linestyle="--", alpha=0.5)
axes[2].set_axisbelow(True)
axes[2].set_xlabel("Country", fontsize=18)
axes[2].tick_params(axis="x", rotation=90)

# alternating background
for i in range(0, len(obs_plot), 2):
    for ax in axes:
        ax.axvspan(i - 0.5, i + 0.5, color="grey", alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig("../Products/country_stats_combined.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\1784754425.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

As table + stats

In [7]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

country_freq = df["Country"].value_counts()

pd.set_option("display.max_rows", None)
print(len(country_freq))
print(len(country_freq[country_freq < 11]))
country_freq


C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\1245431676.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

90
35


Country
Switzerland                 24086
Germany                     12720
Austria                      6247
United Kingdom               5419
France                       2293
Canada                       2175
United States of America     1991
Estonia                      1681
Spain                        1611
Italy                        1431
Ireland                       966
Chile                         947
Netherlands                   926
Kyrgyzstan                    785
Costa Rica                    629
Greece                        612
Cyprus                        542
Portugal                      410
Australia                     357
Sweden                        249
Guatemala                     237
Hungary                       182
Malaysia                      156
Norway                        153
Brazil                        148
Malta                         142
Indonesia                     124
Argentina                     110
Slovakia                      110
Philip

As table

In [8]:
pd.set_option("display.max_rows", None)
users_per_country.sort_index(ascending=True)

Country
Afghanistan                    1
Albania                        1
Antarctica                     2
Argentina                     19
Australia                     14
Austria                       95
Bangladesh                     2
Belgium                        6
Brazil                        46
Bulgaria                       3
Cambodia                       2
Cameroon                       1
Canada                        47
Chad                           1
Chile                         46
China                          8
Colombia                       6
Costa Rica                   181
Croatia                        6
Cyprus                         4
Czechia                       12
Denmark                        3
Ecuador                       23
Egypt                          2
El Salvador                    7
Estonia                        7
Ethiopia                       4
Finland                        4
France                        77
Georgia                        1
Ge

As table

In [9]:
obs_per_user_country.sort_values(ascending=False)

Country
Estonia                     240.142857
Cyprus                      135.500000
Austria                      65.757895
Spain                        57.535714
Canada                       46.276596
Malta                        35.500000
United Kingdom               34.515924
Portugal                     34.166667
France                       29.779221
Guatemala                    29.625000
Serbia                       28.000000
Germany                      25.748988
Australia                    25.500000
Greece                       25.500000
Kyrgyzstan                   23.088235
Ireland                      23.000000
Italy                        20.739130
Chile                        20.586957
Malaysia                     19.500000
Slovakia                     18.333333
United States of America     13.544218
Russia                       13.333333
Hungary                      13.000000
Switzerland                  12.486262
Lebanon                      12.285714
Indonesia        

More bar charts. Here, for the number of observations per user. Since there are a lot of users, only the ones with 50 or more observations are shown.

In [10]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

User_freq = df["created_by"].value_counts()

top100 = User_freq.head(100)
other = pd.Series({"Other": User_freq.iloc[100:].sum()})
User_freq_plot = pd.concat([top100, other])

ax = User_freq_plot.plot(kind="bar", color="teal", figsize=(16,9), logy = True, fontsize=12)

plt.title("Number of Observations per User", fontsize=20)
plt.ylabel("Number of Observations", fontsize=17)
plt.xticks(rotation=90)

#for p in ax.patches: # absolute counts above bars
#    ax.annotate(
#        f'{int(p.get_height())}', 
#        (p.get_x() + p.get_width() / 2., p.get_height()), 
#        ha='center', 
#        va='bottom', 
#        fontsize=10,
#        xytext=(0, 5),
#        textcoords='offset points',
#        rotation=90
#    )

ax.annotate( # bracket
    "",
    xy=(0.492, -0.03),
    xytext=(0.9, -0.03),
    xycoords="axes fraction",
    textcoords="axes fraction",
    arrowprops=dict(arrowstyle="-[, widthB=43.85, lengthB=0.5, angleB=270", color="black", lw=1.5)
)

ax.plot( # line connecting to bracket
    [0.492, 0.492],
    [-0.03, -0.05],
    color="black", 
    lw=1.5, 
    transform=ax.transAxes,
    clip_on=False
)

ax.text( # label for bracket
    0.492,
    -0.08,
    "Individual Users",
    transform=ax.transAxes,
    fontsize=12,
    ha="center"
)

ax.grid(axis="y", linestyle="--", alpha=0.5, which="both")
ax.set_axisbelow(True)
ax.set_ylim(0,30000)
ax.set_xticks([])
ax.set_xticks([len(User_freq_plot) - 1])
ax.set_xticklabels(["Other users"], fontsize=12)

plt.savefig(f"../Products/obs_per_User.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\2630018885.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

How many users in total? How many users with ten or less?

In [20]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

User_freq = df["created_by"].value_counts()

print(len(User_freq))
print(len(User_freq[User_freq < 6]))
print(User_freq[User_freq > 700].sum()/User_freq.sum())
User_freq

C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\2332184767.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

3749
2916
0.5039512436940483


created_by
28919     3841
35124     3764
30178     3546
95692     2406
29542     1992
3065      1967
2316      1805
15185     1762
27610     1647
52314     1647
6542      1465
12938     1371
17235     1263
7079      1087
7867       939
16741      919
64828      837
82037      810
11370      769
2267       727
66748      689
29254      681
11290      539
476        517
109461     457
28674      455
17059      453
77721      427
31773      415
33894      415
13032      399
68724      389
15038      387
39422      377
25925      326
2005       322
23813      300
93344      295
2203       282
38002      281
11175      279
14034      274
52028      266
49909      257
31758      249
1996       246
111531     223
56020      216
11596      197
37254      196
15989      184
36126      171
6529       162
16176      155
95590      155
94379      147
32346      142
62827      140
65591      140
121939     140
95596      136
14100      134
65505      133
40769      132
2270       130
95588      113

Here, an overview of the users is created. More precisely, the number of users that made 1, 2-5, 6-10, 11-50, 51-100, 101-1000 and more than 1000 observations.

In [12]:
# uses User_freq

c1 = 0
c5 = 0
c10 = 0
c50 = 0
c100 = 0
c1000 = 0
c10000 = 0
for count in User_freq:
    if count == 1:
        c1 += 1
    elif count <= 5:
        c5 += 1
    elif count <= 10:
        c10 += 1
    elif count <= 50:
        c50 += 1
    elif count <= 100:
        c100 += 1
    elif count <= 1000:
        c1000 += 1
    else:
        c10000 += 1

bins = ["1 observation","2-5 observations","6-10 observations","11-50 observations", "51-100 observations","101-1000 observations",">1000 observations"]
counts = [c1,c5,c10,c50,c100,c1000,c10000]
bin_series = pd.Series(counts, index=bins)
total = c1+c5+c10+c50+c100+c1000+c10000

plt.figure(figsize=(16,9))
ax = bin_series.plot(kind="bar", color="teal", fontsize=12)

plt.title("Number of Users by Observation Count", fontsize=20)
plt.xlabel("Number of Observations per User", fontsize=17)
plt.ylabel("Number of Users", fontsize=17)
plt.xticks(rotation=90)

# Percentages above bars
#for p in ax.patches:
#    ax.annotate(
#        f'{round(int(p.get_height())/total*100,2)}%',
#        (p.get_x() + p.get_width()/2., p.get_height()),
#        ha='center',
#        va='bottom',
#        xytext=(0,5),
#        textcoords='offset points'
#    )

ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

plt.savefig(f"../Products/Users_per_obs_count.png", dpi=300, bbox_inches="tight")
plt.close()

Kernel Density Estimation to look at the density of observations

In [21]:
df = pd.read_csv("../CWData_clean7.csv")
df = df[(df["latitude"].between(-90, 90)) & (df["longitude"].between(-180, 180))]
world = gpd.read_file(f"../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")

# KDE
xy = np.vstack([df["longitude"], df["latitude"]])
kde = gaussian_kde(xy, bw_method=0.05)  # change bw_method for more/less smooting

# grid
lon_grid, lat_grid = np.meshgrid(
    np.linspace(-180, 180, 800),
    np.linspace(-90, 90, 400)
)
grid_coords = np.vstack([lon_grid.ravel(), lat_grid.ravel()])

z = kde(grid_coords).reshape(lon_grid.shape)
z_log = np.log1p(z)  # log scale, else we only see Europe
z_masked = np.where(z_log < 1e-5, np.nan, z_log) # do not show pixels without anything

fig, ax = plt.subplots(figsize=(16, 8), subplot_kw={"projection": ccrs.Robinson()})
ax.set_global()

ax.add_geometries(
    world.geometry,
    crs=ccrs.PlateCarree(),  # SHP is in WGS84
    facecolor="grey",
    edgecolor="grey",
    linewidth=0.5,
    zorder=1
)

# plot KDE
colors = ["#ffffff00", "#4393c3", "#2166ac", "#d6604d", "#b2182b"]
cmap = mcolors.LinearSegmentedColormap.from_list("cw_kde", colors)

vmin, vmax = np.nanmin(z_masked), np.nanpercentile(z_masked, 99) # clip highest percent to make rest of the world more colorful (not just Switzerland)

mesh = ax.pcolormesh(
    lon_grid, lat_grid, z_masked,
    transform=ccrs.PlateCarree(),
    cmap=cmap,
    shading="auto",
    zorder=2,
    alpha=0.85,
    vmax=vmax,
    vmin=vmin
)

ax.add_geometries(
    world.geometry,
    crs=ccrs.PlateCarree(),
    facecolor="none",
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

ticks = [vmin, vmin + (vmax-vmin)*0.25, vmin + (vmax-vmin)*0.5, vmin + (vmax-vmin)*0.75, vmax]
labels = ["Low", "", "Medium", "", "High"]

cbar = plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.03, fraction=0.03)
cbar.set_label("Density (Log Scale)", fontsize=15)
cbar.set_ticks(ticks)
cbar.set_ticklabels(labels)
cbar.ax.tick_params(labelsize=12)

ax.set_title("Global Density of Observations using Kernel Density Estimation (KDE)", fontsize=18)

plt.savefig("../Products/KDE_global.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\1263467607.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

Again, density: Hexbin

In [22]:
df = pd.read_csv("../CWData_clean7.csv")
df = df[(df["latitude"].between(-90, 90)) & (df["longitude"].between(-180, 180))]
world = gpd.read_file("../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")

fig, ax = plt.subplots(figsize=(16, 8), subplot_kw={"projection": ccrs.Robinson()})
ax.set_global()

# background
ax.add_geometries(
    world.geometry,
    crs=ccrs.PlateCarree(),
    facecolor="grey",
    edgecolor="grey",
    linewidth=0.5,
    zorder=1
)

# transform coordinates to Robinson
transformer = ccrs.Robinson().transform_points(
    ccrs.PlateCarree(),
    df["longitude"].values,
    df["latitude"].values
)
x_rob = transformer[:, 0]
y_rob = transformer[:, 1]

# Hexbin
cmap = plt.cm.YlOrRd
cmap.set_under(color="none")

hb = ax.hexbin(
    x_rob, y_rob,
    gridsize=100,
    bins="log",
    cmap=cmap,
    mincnt=1,
    zorder=2,
    transform=ccrs.Robinson(),
    linewidths=0
)

# country borders
ax.add_geometries(
    world.geometry,
    crs=ccrs.PlateCarree(),
    facecolor="none",
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

cbar = plt.colorbar(hb, ax=ax, orientation="horizontal", pad=0.03, fraction=0.03)
cbar.set_label("Number of Observations (Log Scale)", fontsize=15)
cbar.ax.tick_params(labelsize=12)
ax.set_title("Global Distribution of Observations (Hexbin)", fontsize=18)

plt.savefig("../Products/Hexbin_global.png", dpi=300, bbox_inches="tight")
plt.close()

C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\3507665843.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

Then, the number of total observations per 1'000 km2 for each country is plotted on a map.

In [23]:
df = pd.read_csv("../CWData_clean7.csv")
df["created_at_local"] = pd.to_datetime(df["created_at_local"])

# dataframe with country and the number of observations in it
obs_per_country = (
    df
    .groupby("Country")
    .agg(
        total_observations = ("created_at_local","count"),
    )
    .reset_index()
)

pd.set_option("display.max_rows", None)

world = gpd.read_file(f"../Borders/ne_10m_admin_0_countries/ne_10m_admin_0_countries.shp", engine="fiona") # higher resolution map because of area accuracy
world = world.rename(columns={"NAME": "Country"})
world_area = gpd.read_file(f"../Borders/ne_10m_admin_0_countries/ne_10m_admin_0_countries.shp", engine="fiona") # higher resolution map because of area accuracy
world_area = world_area.to_crs("ESRI:54009") # area-true projection, Robinson skews it --> used for area calculations
obs_per_country["ISO_A3"] = obs_per_country["Country"].apply(own.get_iso3)

map_df = world.merge(obs_per_country, left_on="ADM0_A3", right_on="ISO_A3", how="left").to_crs("+proj=robin")
map_df_area = world_area.merge(obs_per_country, left_on="ADM0_A3", right_on="ISO_A3", how="left")

bins = [0,0.01,0.1,1,5,50,float("inf")]
labels = ["<0.01","0.01-0.1","0.11-1","1-5","5-50",">50"]
map_df_area["area_km2"] = map_df_area.geometry.area / 1e6 # area from map_df_area
map_df["area_km2"] = map_df_area["area_km2"].values # area transferred to map_df
map_df["obs_per_1000km2"] = map_df["total_observations"] / map_df["area_km2"] * 1000
map_df["cat_obs_per_1000km2"] = pd.cut(map_df["total_observations"] / map_df["area_km2"] * 1000, bins=bins, labels=labels)

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

cmap = get_cmap("Reds", len(labels))
legend_handles = [
    Patch(facecolor=cmap(i / (len(labels) - 1)), edgecolor="black", linewidth=0.5, label=labels[i])
    for i in range(len(labels))
]
legend_handles.append(Patch(facecolor="grey", edgecolor="black", linewidth=0.5, label="No observations"))

map_df.plot(
    column="cat_obs_per_1000km2",
    cmap="Reds",
    linewidth=0.5,
    edgecolor="black",
    missing_kwds={"color": "grey"},
    legend=False,
    categorical=True,
    ax=ax
)

ax.legend(handles=legend_handles, title="Obs/1000km²", fontsize=12, title_fontsize=15, loc="lower left")
ax.set_title("Number of Observations per 1000 km²", fontsize=18)
ax.set_axis_off()

plt.savefig(f"../Products/number_of_obs_per_1000km2_per_Country.png", dpi=300, bbox_inches="tight")
plt.close()
map_df.to_csv("../Products/CSVs/obs_per_1000kms.csv", index=False)

C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\2481036292.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 

In [25]:
map_df[["Country_x", "obs_per_1000km2", "cat_obs_per_1000km2", "area_km2", "total_observations"]].sort_values("obs_per_1000km2", ascending=False)

,Country_x,obs_per_1000km2,cat_obs_per_1000km2,area_km2,total_observations
91,Switzerland,581.525823,>50,4.141862e+04,24086.0
222,Malta,435.089074,>50,3.263700e+02,142.0
94,Liechtenstein,182.237532,>50,1.371836e+02,25.0
7,Cyprus,100.229306,>50,5.407600e+03,542.0
88,Austria,74.419137,>50,8.394346e+04,6247.0
50,Estonia,36.800919,5-50,4.567821e+04,1681.0
49,Germany,35.612619,5-50,3.571768e+05,12720.0
93,Netherlands,24.801113,5-50,3.733703e+04,926.0
78,United Kingdom,22.274704,5-50,2.432804e+05,5419.0
57,Luxembourg,19.956978,5-50,2.605605e+03,52.0


Now, I will also look at the spatial autocorrelation

In [28]:
df = pd.read_csv("../CWData_clean7.csv")
df["year_month"] = pd.to_datetime(df["year_month"]).dt.to_period("M")
df["year"] = df["year_month"].dt.year
df["lon_bin"] = np.floor(df["longitude"]).astype(int)
df["lat_bin"] = np.floor(df["latitude"]).astype(int)

world = gpd.read_file("../Borders/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp", engine="fiona")

# create full grid with 1° grid cells
lon_bins = np.arange(-180, 180)
lat_bins = np.arange(-90, 90)
grid = pd.MultiIndex.from_product([lat_bins, lon_bins], names=["lat_bin", "lon_bin"]).to_frame(index=False)

all_years = sorted(df["year"].unique())

# LISA colors
lisa_colors = {
    "HH": "red",  # High-High Hotspot
    "HL": "orange",  # High-Low Outlier
    "LH": "lightblue",  # Low-High Outlier
    "ns": "white"   # non-significant
}

# observations per grid cell
counts = (
    df
    .groupby(["lat_bin", "lon_bin"])
    .size()
    .reset_index(name="n_obs")
)

grid_year = grid.merge(counts, on=["lat_bin", "lon_bin"], how="left").fillna(0)
grid_active = grid_year[grid_year["n_obs"] > 0].copy() # ignore cells without any observations

grid_active["geometry"] = grid_active.apply(
    lambda row: box(row["lon_bin"], row["lat_bin"], row["lon_bin"] + 1, row["lat_bin"] + 1),
    axis=1
)
gdf_active = gpd.GeoDataFrame(grid_active, geometry="geometry", crs="EPSG:4326")

gdf_metric = gdf_active.to_crs("EPSG:3857") # metric CRS for distance-based neighborhood (because else neighors might be very far away because of empty cells)

w = DistanceBand.from_dataframe(gdf_metric, threshold=500000, silence_warnings=True)
isolated = [i for i, neighbors in w.neighbors.items() if len(neighbors) == 0]
gdf_metric = gdf_metric.drop(index=isolated).reset_index(drop=True)

w = DistanceBand.from_dataframe(gdf_metric, threshold=500000, silence_warnings=True)
w.transform = "r"

y = gdf_metric["n_obs"].values

# seed makes it reproducible
np.random.seed(1)

# calculate LISA
lisa = Moran_Local(y, w, permutations=999, seed=1)

# calculate Moran's I globally
moran = Moran(y, w, permutations=999)

# significance: p < 0.05
sig = lisa.p_sim < 0.1
quads = lisa.q  # 1=HH, 2=LH, 3=LL, 4=HL
quad_map = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}

gdf_metric["lisa_cat"] = "ns"
gdf_metric.loc[sig, "lisa_cat"] = [quad_map[q] for q in quads[sig]]

gdf_metric = gdf_metric.to_crs("+proj=robin")
gdf_sig = gdf_metric[gdf_metric["lisa_cat"] != "ns"] # only plot significant cells

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

world.to_crs("+proj=robin").plot(
    ax=ax, color="grey", edgecolor="black", linewidth=0.5, zorder=1
)

for cat, color in lisa_colors.items():
    if cat == "ns":
        continue
    subset = gdf_sig[gdf_sig["lisa_cat"] == cat]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, zorder=2, alpha=0.8)

patches = [
    mpatches.Patch(facecolor=lisa_colors["HH"], edgecolor="black", linewidth=0.5, label="High-High (Hotspot)"),
    mpatches.Patch(facecolor=lisa_colors["HL"], edgecolor="black", linewidth=0.5, label="High-Low (Outlier)"),
    mpatches.Patch(facecolor=lisa_colors["LH"], edgecolor="black", linewidth=0.5, label="Low-High (Outlier)"),
]
legend = ax.legend(handles=patches, title="LISA Cluster (α = 0.1)", loc="lower left", frameon=True)
legend.get_title().set_fontsize(15)
for text in legend.get_texts():
    text.set_fontsize(12)

ax.set_title(f"Local Indicators of Spatial Autocorrelation (LISA)", fontsize=18)

# write Moran's I on map
ax.text(
    0.011, 0.93,
    f"Moran's I = {moran.I:.3f} (p = {moran.p_sim:.3f})",
    transform=ax.transAxes,
    fontsize=12,
    verticalalignment="bottom",
    bbox=dict(boxstyle="round", facecolor="white")
)

ax.set_axis_off()

# Inset for Europe
ax_inset = fig.add_axes([0.129, 0.245, 0.2, 0.3])  # [left, bottom, width, height] in figure fraction

world.to_crs("+proj=robin").plot(
    ax=ax_inset, color="grey", edgecolor="black", linewidth=0.5, zorder=1
)

for cat, color in lisa_colors.items():
    if cat == "ns":
        continue
    subset = gdf_sig[gdf_sig["lisa_cat"] == cat]
    if len(subset) > 0:
        subset.plot(ax=ax_inset, color=color, zorder=2, alpha=0.8)

# extent of Europe
ax_inset.set_xlim(-1000000, 2500000)
ax_inset.set_ylim(3700000, 6500000)
ax_inset.set_axis_off()
ax_inset.set_title("Europe", fontsize=15)

ax_inset.set_axis_off()

# border of inset
rect = Rectangle((0, 0), 1, 1, transform=ax_inset.transAxes,
                 fill=False, edgecolor="black", linewidth=2, zorder=10)
ax_inset.add_patch(rect)

plt.savefig(f"../Products/LISA.png", dpi=300, bbox_inches="tight")
plt.close()


C:\Users\yanni\AppData\Local\Temp\ipykernel_31884\3671233021.py:1: DtypeWarning: Columns (0: Plastic_Amount, 1: Plastic_Location, 2: Plastic_RiverWidth, 3: Plastic_PET, 4: Plastic_POSoft, 5: Plastic_POHard, 6: Plastic_PS, 7: Plastic_PSE, 8: Plastic_PMultilayer, 9: Plastic_POther, 10: Plastic_Shore_Plotsize, 11: Plastic_Removed, 12: Plastic_River_Stagnant, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: StreamColor_other, 28: Stream_Vegetation, 29: Stream_Foam, 30: Stream_Algae, 31: Stream_Odor, 32: Stream_Odor_Type, 33: Stream_Odor_Type_other, 34: Stream_Litter, 35: Stream_Flow_Alteration, 36: Stream_typical_Color, 37: Stream_Drainage_Basin, 38: Lake_Usage, 39: Lake_Usage_Nr, 40: Lake_Access, 41: Lake_Shore_State, 